In [2]:
pip install fastapi uvicorn[standard] pydantic


  Using cached fastapi-0.116.1-py3-none-any.whl.metadata (28 kB)
  Using cached pydantic-2.11.7-py3-none-any.whl.metadata (67 kB)
  Using cached uvicorn-0.35.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached click-8.2.1-py3-none-any.whl.metadata (2.5 kB)
  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.1-py3-none-any.whl.metadata (2.6 kB)
Using cached fastapi-0.116.1-py3-none-any.whl (95 kB)
Using cached pydantic-2.11.7-py3-none-any.whl (444 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---

In [4]:
import os
os.chdir("C:/Users/dedee/Downloads/epidemic-project")
print("Now working from:", os.getcwd())


Now working from: C:\Users\dedee\Downloads\epidemic-project


In [5]:
from src.predictors import load_panel, load_adj, load_models, lstm_multi_step_forecast, gibbs_multi_day_rollout, aggregate_trajectories_to_forecasts


ModuleNotFoundError: No module named 'src.predictors'

In [2]:
# src/api.py
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import pandas as pd
from src.predictors import load_panel, load_adj, load_models, lstm_multi_step_forecast, gibbs_multi_day_rollout, aggregate_trajectories_to_forecasts
import numpy as np


app = FastAPI(title="Epidemic Forecast API")

# load resources once
panel = load_panel()
adj = load_adj()
lstm, clf, scaler = load_models()

class ForecastRequest(BaseModel):
    state: str
    horizon: int = 7
    ensemble: int = 200

@app.get("/")
def root():
    return {"message":"Epidemic Forecast API"}

@app.post("/predict")
def predict(req: ForecastRequest):
    state = req.state
    H = req.horizon
    M = req.ensemble

    # prepare Z_prev (last observed day)
    last_day = panel['Date'].max()
    last_rows = panel[panel['Date']==last_day]
    Z_prev = dict(zip(last_rows['state'], (last_rows['active']>0).astype(int)))

    # prepare feature maps and lstm inputs
    # build X_seq for each state from panel: latest seq_len rows (must match seq_len used in LSTM)
    seq_len = 14
    features = ['active','new_confirmed','daily_tests','daily_vax','vax_rate']
    lstm_preds_by_state = {}
    lstm_means_by_state = {}
    features_map = {}
    states = last_rows['state'].tolist()
    for st in states:
        sub = panel[panel['state']==st].sort_values('Date').reset_index(drop=True)
        recent = sub.tail(seq_len)
        if len(recent) < seq_len:
            # pad by repeating last
            pad = np.vstack([recent[features].values] * (seq_len - len(recent)))
            X_seq = np.vstack([pad, recent[features].values])
        else:
            X_seq = recent[features].values
        region_id = int(recent['state_id'].iloc[-1])
        preds = lstm_multi_step_forecast(lstm, X_seq, region_id, steps=H)
        lstm_preds_by_state[st] = preds
        lstm_means_by_state[st] = preds  # here treat model output as mean; if scaled, convert appropriately
        features_map[st] = {'daily_tests': int(recent['daily_tests'].iloc[-1]) if 'daily_tests' in recent.columns else 0,
                            'daily_vax': int(recent['daily_vax'].iloc[-1]) if 'daily_vax' in recent.columns else 0}

    # run Gibbs to create trajectories
    trajectories = gibbs_multi_day_rollout(adj, clf, scaler, Z_prev, lstm_preds_by_state, features_map, H=H, M=M)
    forecasts = aggregate_trajectories_to_forecasts(trajectories, lstm_means_by_state, alpha=0.3, phi=20, cfr=0.01)

    # prepare response for requested state
    res = {'state': state, 'horizon': H, 'forecast': []}
    for day in range(1, H+1):
        if state in forecasts[day]:
            res['forecast'].append({'day': day, **forecasts[day][state]})
        else:
            res['forecast'].append({'day': day, 'error': 'state not found'})
    return res

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)


ModuleNotFoundError: No module named 'src'

In [3]:
import os

# show current working directory
print("Current directory:", os.getcwd())

# list all files/folders in it
for f in os.listdir():
    print(f)


Current directory: C:\Users\dedee\Downloads\epidemic-project\src
.ipynb_checkpoints
adjacency.ipynb
api.ipynb
data_curated
eda.ipynb
etl.ipynb
evaluate.ipynb
features.ipynb
gibbs.ipynb
lstm_model.ipynb
lstm_preprocess.ipynb
markov.ipynb
models
predictors.ipynb
prophet_baseline.ipynb
results
